# Atelier Scikit-learn — Prédiction de l'état d'un capteur IoT

**Contexte.** Des capteurs mesurent température, humidité, pression et consommation. Chaque mesure a un **état** : `OK`, `ALERTE` ou `ERREUR`. On veut un modèle qui **prédit automatiquement l'état** d'une mesure à partir de ses valeurs numériques.

**Workflow ML classique suivi :**
Dataset → Chargement → Exploration → Nettoyage → X / y → Train/Test → Prétraitement → Modèle → `fit()` → `predict()` → Évaluation → Sauvegarde → Chargement → Réutilisation.

## Partie 0 — Mise en place de l'environnement

On importe **pandas** (manipulation de données), **matplotlib** et **seaborn** (visualisation). Le CSV se trouve dans `data/` ; le notebook s'exécutant depuis `notebooks/`, on remonte d'un niveau (`../data/...`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/mesures_capteurs.csv")
print("Forme du dataframe :", df.shape)
df.head()

### Exploration du dataframe

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Repartition de la variable cible 'etat'
df["etat"].value_counts(dropna=False)

**Observation importante.** Les classes sont **très déséquilibrées** : la grande majorité des mesures sont `OK`, `ALERTE` est rare et `ERREUR` très rare. Ce déséquilibre aura des conséquences sur l'évaluation (Partie 7).

## Partie 1 — Gestion des doublons

Un **doublon** est une ligne strictement identique à une autre. On le repère avec `duplicated()` puis on le supprime avec `drop_duplicates()`.

In [ ]:
nb_doublons = df.duplicated().sum()
print("Nombre de doublons :", nb_doublons)

In [ ]:
df = df.drop_duplicates()
print("Doublons apres suppression :", df.duplicated().sum())
print("Nouvelle forme :", df.shape)

### Nettoyage nécessaire : lignes sans état

La cible `etat` contient quelques valeurs manquantes. Or **on ne peut ni entraîner ni évaluer** un modèle sur des lignes dont la réponse est inconnue. On supprime donc les lignes où `etat` est manquant (les valeurs manquantes **des caractéristiques**, elles, seront traitées proprement en Partie 4).

In [ ]:
print("Etats manquants avant :", df["etat"].isna().sum())
df = df.dropna(subset=["etat"])
print("Etats manquants apres :", df["etat"].isna().sum())
print("Forme du dataframe :", df.shape)

## Partie 2 — Sélection de `y` (cible) et `X` (caractéristiques)

- **cible** `y` = `etat` (ce qu'on veut prédire),
- **caractéristiques** `X` = `temperature`, `humidite`, `pression`, `consommation` (les variables explicatives numériques).

In [ ]:
caracteristiques = ["temperature", "humidite", "pression", "consommation"]
X = df[caracteristiques]
y = df["etat"]

print("X (5 premieres lignes) :")
print(X.head())
print("\ny (5 premieres valeurs) :")
print(y.head())

### Type de problème de Machine Learning

La cible `etat` prend des valeurs **catégorielles** (`OK`, `ALERTE`, `ERREUR`). Il s'agit donc d'un problème de **classification** — plus précisément de **classification multi-classe** (3 classes).

## Partie 3 — Découpage Train / Test

Conditions : 20 % pour le test, reproductibilité (`random_state`), et **`stratify=y`** pour conserver **les mêmes proportions de classes** dans le train et le test que dans les données d'origine — indispensable ici vu le déséquilibre des classes.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train :", X_train.shape, " | Test :", X_test.shape)
print("\nProportions train :\n", y_train.value_counts(normalize=True).round(3))
print("\nProportions test :\n", y_test.value_counts(normalize=True).round(3))

## Partie 4 — Gestion des valeurs manquantes

`SimpleImputer` remplace chaque valeur manquante par une valeur calculée. On choisit la **médiane**.

In [ ]:
print("Valeurs manquantes par colonne (X_train) :")
print(X_train.isna().sum())

**Pourquoi la médiane ?** La médiane est **robuste aux valeurs extrêmes (outliers)**. Contrairement à la moyenne, qui est tirée vers le haut ou le bas par quelques valeurs aberrantes, la médiane (valeur centrale) reste représentative. Sur des données de capteurs qui peuvent avoir des pics anormaux, c'est le choix prudent.

**Point méthodologique clé :** on calcule les médianes **uniquement sur `X_train`** (`fit`), puis on applique la transformation au train **et** au test (`transform`). Cela évite la *fuite de données* (data leakage) : le test doit rester "inconnu".

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")
imputer.fit(X_train)                       # apprend les medianes sur le train

print("Medianes apprises :", dict(zip(caracteristiques, imputer.statistics_.round(2))))

X_train_imputed = imputer.transform(X_train)
X_test_imputed  = imputer.transform(X_test)
print("\nValeurs manquantes restantes (train) :", np.isnan(X_train_imputed).sum())

## Partie 5 — Mise à l'échelle (standardisation)

**Pourquoi standardiser ?** Le modèle KNN se base sur des **distances** entre points. Or nos variables ont des échelles très différentes (pression ~1000, température ~25). Sans mise à l'échelle, la pression **écraserait** les autres variables dans le calcul de distance. `StandardScaler` recentre chaque variable (moyenne 0) et la réduit (écart-type 1), pour que **toutes pèsent équitablement**.

Là encore : `fit` sur le **train** seulement, puis `transform` sur train et test.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train_imputed)                # apprend moyennes et ecarts-types sur le train

print("Moyennes  :", scaler.mean_.round(2))
print("Ecarts-types :", scaler.scale_.round(2))

X_train_scaled = scaler.transform(X_train_imputed)
X_test_scaled  = scaler.transform(X_test_imputed)

## Partie 6 — Entraînement et prédiction (KNN)

**KNN (k plus proches voisins)** classe une nouvelle mesure en regardant les **k=5** points d'entraînement les plus proches et en prenant la classe **majoritaire** parmi eux.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)           # entrainement

y_pred = knn.predict(X_test_scaled)        # prediction sur le test
print("Quelques predictions :", y_pred[:15])

### Comparaison prédictions / vraies valeurs

In [ ]:
comparaison = pd.DataFrame({
    "vraie_valeur": y_test.values,
    "prediction": y_pred
})
comparaison["correct"] = comparaison["vraie_valeur"] == comparaison["prediction"]
print(comparaison.head(15))
print("\nBonnes predictions :", comparaison["correct"].sum(), "/", len(comparaison))

## Partie 7 — Évaluation du modèle

In [ ]:
from sklearn.metrics import accuracy_score

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy : {acc:.3f}")

### Pourquoi l'accuracy ne suffit pas toujours ?

L'**accuracy** = proportion de prédictions correctes. Le piège : avec des classes **déséquilibrées**, un modèle qui prédit **toujours `OK`** obtiendrait déjà une accuracy très élevée (puisque presque tout est `OK`), **sans jamais détecter** une `ALERTE` ou une `ERREUR` — qui sont pourtant les cas les plus importants ! L'accuracy masque donc les mauvaises performances sur les classes rares.

### Matrice de confusion

Elle croise **vraies valeurs (lignes)** et **prédictions (colonnes)**. La diagonale = bonnes prédictions ; hors diagonale = erreurs.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

classes = sorted(y.unique())               # ['ALERTE', 'ERREUR', 'OK']
cm = confusion_matrix(y_test, y_pred, labels=classes)
print("Classes :", classes)
print("Matrice de confusion :\n", cm)

### Visualisation de la matrice avec Seaborn

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=classes, yticklabels=classes)
plt.xlabel("Prediction")
plt.ylabel("Vraie valeur")
plt.title("Matrice de confusion - KNN")
plt.show()

**Commentaire.** Le modèle classe très bien les `OK` (classe majoritaire). En revanche il se trompe souvent sur `ALERTE` et surtout `ERREUR`, faute d'assez d'exemples pour ces classes à l'entraînement — conséquence directe du **déséquilibre**.

### Rapport de classification

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred, zero_division=0))

**Commentaire.** Le rapport donne **precision**, **recall** et **f1-score** *par classe*. On voit que les scores sont bons pour `OK` mais faibles pour les classes rares : le modèle est bon "en moyenne" mais peu fiable pour détecter les anomalies.

### Quand privilégier chaque métrique ?

- **Precision** — quand le **coût d'une fausse alerte est élevé**. Elle répond à : "parmi ce que j'ai prédit `ALERTE`, combien l'étaient vraiment ?" (ex. éviter de déclencher une intervention coûteuse pour rien).
- **Recall (rappel)** — quand **rater un cas positif est grave**. "parmi les vraies `ERREUR`, combien ai-je détectées ?" (ex. détection de panne : mieux vaut une fausse alerte que rater une vraie erreur).
- **F1-score** — quand on veut un **compromis** entre precision et recall (moyenne harmonique). Utile justement en cas de **classes déséquilibrées**.
- **Accuracy** — appropriée quand les **classes sont équilibrées** et que toutes les erreurs ont la même importance.